In [1]:
# Установка необходимых библиотек для Kaggle (запустите эту ячейку первой)
# Установка необходимых библиотек для Kaggle (обновлённая: добавлен mapclassify для geopandas.explore())
# !pip install osmnx pystac-client folium stackstac xarray-spatial geopandas mapclassify -q
# !pip install -q rioxarray
!pip install -q pystac-client rioxarray geopandas osmnx folium rasterio mapclassify scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 194.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 60.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 113.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 114.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 114.3 kB/s eta 0:00:00


In [2]:
# ======================================================================================
# FLOOD MAP — ПОЛНЫЙ ПАЙПЛАЙН (загрузка → анализ → карта)
# Версия 2.1 — включает загрузку тайлов ArcticDEM
# ======================================================================================
#
# ЗАПУСК НА KAGGLE:
# 1. Создайте новый Notebook
# 2. Первая ячейка:
#    !pip install -q pystac-client rioxarray geopandas osmnx folium rasterio mapclassify scipy
# 3. Вторая ячейка: вставьте весь этот файл
#
# ИСПРАВЛЕНИЯ ОТНОСИТЕЛЬНО ОРИГИНАЛА:
# 1. Полный пайплайн: загрузка тайлов + анализ + карта
# 2. Гидрологическая связность (flood fill от seed-точек на воде)
# 3. Калибровка по маске воды из OSM (не по перцентилю)
# 4. Упрощение геометрии в метрической CRS (не в градусах!)
# 5. Фильтрация мелких полигонов
# 6. Морфологическая очистка маски
# 7. Анализ зданий в зоне затопления
# 8. Совместимость с osmnx >= 2.0

import os
import glob
import numpy as np
import geopandas as gpd
import rioxarray
import folium
from rasterio.features import shapes, rasterize
from shapely.geometry import box, Point
from scipy import ndimage
from osgeo import gdal
import pystac_client
import warnings

warnings.filterwarnings("ignore")
gdal.UseExceptions()


# ======================================================================================
# ПАРАМЕТРЫ (единственное место, которое нужно менять)
# ======================================================================================
CITY_NAME = "Saint Petersburg, Russia"
FLOOD_LEVEL = 3.0             # Уровень воды в системе БС-77 (метры).
# Реальный порог DSM = FLOOD_LEVEL + water_offset, где water_offset — медиана
# WGS84-высот на воде (~16.09м). Итого: ~19.1м WGS84 = порог затопления.
CRS_METRIC = "EPSG:32636"     # UTM Zone 36N (подходит для СПб)
OUTPUT_DIR = "/kaggle/working/"
TARGET_RES = 2               # Разрешение для анализа (метры)
MIN_POLYGON_AREA_M2 = 50     # Минимальная площадь полигона затопления
SIMPLIFY_TOLERANCE_M = 5     # Упрощение геометрии (метры, в метрической CRS!)
MORPH_CLOSING_RADIUS = 1      # Радиус морфологического закрытия (пиксели × TARGET_RES).
# При TARGET_RES=2м: 1px=2м, 3px=6м.
# Значение 3 заполняет дороги (6м) → ложные связи. Рекомендуется 1-2.

# ArcticDEM 2m mosaic tiles for Saint Petersburg (29–31°E, 59–60°N)
# Tile naming: {lon}_{lat}_{sx}_{sy} — 1°x1° cell divided into 2×2 subtiles
# !!! БЫЛО: 32_73 (73°N = Новая Земля, НЕПРАВИЛЬНО) → ИСПРАВЛЕНО на 59–60°N, 29–30°E
TILE_SHORT_IDS = [
    "29_59_1_1", "29_59_2_1", "29_59_1_2", "29_59_2_2",
    "30_59_1_1", "30_59_2_1", "30_59_1_2", "30_59_2_2",
    "29_60_1_1", "29_60_2_1", "29_60_1_2", "29_60_2_2",
    "30_60_1_1", "30_60_2_1", "30_60_1_2", "30_60_2_2",
]
COLLECTION_ID = "arcticdem-mosaics-v4.1-2m"
STAC_API_URL = "https://stac.pgc.umn.edu/api/v1"

# Seed-точки для заливки (lon, lat) — КЛЮЧЕВЫЕ ВОДНЫЕ ОБЪЕКТЫ
# Точки должны лежать на водной поверхности (Нева, Финский залив)
WATER_SEEDS_WGS84 = [
    # Финский залив
    (29.50, 59.97),   # Залив — запад (за КЗС)
    (29.80, 60.00),   # Залив — центр
    (30.00, 59.96),   # Залив — восток
    (30.10, 59.97),   # Залив — порт
    (30.25, 59.95),   # Финский залив (акватория порта)
    # Нева и дельта
    (30.35, 59.94),   # Устье Невы
    (30.40, 59.95),   # Нева — центр
    (30.45, 59.95),   # Нева — восток
    (30.50, 59.95),   # Нева — Смольная
    # Каналы СПб
    (30.32, 59.93),   # Фонтанка / Мойка
    (30.30, 59.95),   # Малая Нева
    # Север (Кронштадт)
    (29.77, 59.99),   # Финский залив — Котлин
]

VRT_PATH = os.path.join(OUTPUT_DIR, "temp.vrt")
INTERMEDIATE_FILE = os.path.join(OUTPUT_DIR, "dem_merged_reprojected.tif")
OUT_FILENAME = os.path.join(OUTPUT_DIR, "spb_flood_map_v2.html")


# ======================================================================================
# 0. ЗАГРУЗКА ТАЙЛОВ ARCTICDEM
# ======================================================================================
def download_dem_tiles(tile_ids, collection_id, api_url, output_dir):
    """
    Скачивает тайлы ArcticDEM мозаики через PGC STAC API.
    
    Логика из Cell 23 второго блокнота, с улучшениями:
    - Два метода поиска (по ID, затем по свойству pgc:tile)
    - Пропуск уже скачанных файлов
    - Обработка ошибок без прерывания
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Настройки AWS для доступа к данным PGC (публичные, без ключей)
    os.environ['AWS_NO_SIGN_REQUEST'] = 'YES'
    os.environ['AWS_REGION'] = 'us-west-2'
    
    full_ids = [f"{tid}_2m_v4.1" for tid in tile_ids]
    
    print(f"   Коллекция: {collection_id}")
    print(f"   Целевые тайлы: {full_ids}")
    
    api = pystac_client.Client.open(api_url)
    
    # Метод 1: поиск по ID
    search = api.search(collections=[collection_id], ids=full_ids)
    found_items = list(search.items())
    
    if not found_items:
        # Метод 2: поиск по свойству pgc:tile
        print("   Поиск по ID не дал результатов. Пробуем по pgc:tile...")
        try:
            search = api.search(
                collections=[collection_id],
                query={"pgc:tile": {"in": tile_ids}}
            )
            found_items = list(search.items())
        except Exception as e:
            print(f"   Ошибка запроса: {e}")

    if not found_items:
        # Метод 3: bbox-поиск по региону СПб
        print("   Пробуем bbox-поиск для СПб (28-32°E, 59-61°N)...")
        try:
            search = api.search(
                collections=[collection_id],
                bbox=[28.0, 59.0, 32.0, 61.0]
            )
            found_items = list(search.items())
            if found_items:
                print(f"   Bbox-поиск нашёл {len(found_items)} тайлов")
                # Фильтруем только нужные тайлы
                found_items = [it for it in found_items
                               if any(tid in it.id for tid in tile_ids)]
                print(f"   После фильтрации по ID: {len(found_items)} тайлов")
        except Exception as e:
            print(f"   Ошибка bbox-поиска: {e}")
    
    if not found_items:
        raise RuntimeError(
            f"Тайлы не найдены через API ({api_url}). "
            f"Проверьте, что PGC STAC доступен и коллекция {collection_id} существует."
        )
    
    print(f"   Найдено тайлов: {len(found_items)}")
    
    downloaded_files = []
    for item in found_items:
        save_path = os.path.join(output_dir, f"{item.id}.tif")
        
        # Пропускаем уже скачанные
        if os.path.exists(save_path):
            print(f"   Уже скачан: {item.id}")
            downloaded_files.append(save_path)
            continue
        
        # Определяем ключ актива
        # ArcticDEM 2m mosaic uses 'dem' key; fallback to any available asset
        ASSET_PRIORITY = ['dem', 'data', 'DEM', 'image']
        asset_key = next((k for k in ASSET_PRIORITY if k in item.assets), None)
        if asset_key is None:
            asset_key = next(iter(item.assets))
            print(f"   Использую asset '{asset_key}' (доступные: {list(item.assets.keys())})")
        href = item.assets[asset_key].href
        
        try:
            print(f"   Загрузка {item.id}...")
            da = rioxarray.open_rasterio(href)
            da.rio.to_raster(save_path, compress='LZW')
            downloaded_files.append(save_path)
            print(f"   Сохранено: {save_path}")
        except Exception as e:
            print(f"   Ошибка загрузки {item.id}: {e}")
    
    if not downloaded_files:
        raise RuntimeError("Ни один тайл не удалось загрузить.")
    
    return downloaded_files


# ======================================================================================
# 1. ОБЪЕДИНЕНИЕ И РЕПРОЕКЦИЯ (GDAL — экономит RAM)
# ======================================================================================
def merge_and_reproject(tif_files, target_crs, target_res, vrt_path, output_path):
    """Объединяет тайлы через VRT и репроецирует через GDAL Warp."""
    print(f"   Тайлов для объединения: {len(tif_files)}")
    for f in tif_files:
        print(f"     - {os.path.basename(f)}")
    
    # VRT — виртуальная мозаика без загрузки в RAM
    print("   Создание VRT...")
    vrt_options = gdal.BuildVRTOptions(resampleAlg='bilinear')
    ds = gdal.BuildVRT(vrt_path, tif_files, options=vrt_options)
    ds.FlushCache()
    ds = None
    
    # Warp — репроекция на уровне GDAL (C++, минимум RAM)
    print(f"   Репроекция в {target_crs}, разрешение {target_res} м...")
    warp_options = gdal.WarpOptions(
        format="GTiff",
        dstSRS=target_crs,
        xRes=target_res,
        yRes=target_res,
        dstNodata=-9999,
        resampleAlg="bilinear",
        creationOptions=["COMPRESS=LZW", "TILED=YES", "BIGTIFF=YES"],
        multithread=True,
        warpMemoryLimit=512,
    )
    ds = gdal.Warp(output_path, vrt_path, options=warp_options)
    ds = None
    
    print("   DEM готов.")
    return output_path


# ======================================================================================
# 2. КАЛИБРОВКА ПО МАСКЕ ВОДЫ
# ======================================================================================
def calibrate_elevation(dem, bbox_wgs84, crs_metric, seeds_wgs84, buffer_km=2.0):
    """
    Находит смещение DEM (WGS84 эллипсоидальная высота) относительно
    текущего уровня воды. Для СПб WGS84 ≈ EGM2008 + 16.09м ≈ BS-77 + 16.09м.
    Возвращает: water_offset ≈ 16.09м. Далее: threshold = FLOOD_LEVEL + water_offset.
    ВНИМАНИЕ: зависит от состояния воды на момент съёмки DEM (2022-03-09 для v9).
    
    ИСПРАВЛЕНИЕ v2.1: запрашивает воду НЕ по всему bbox (100×70 км — таймаут),
    а только вокруг seed-точек с буфером 2 км. Это в ~500 раз меньше
    по площади и отрабатывает за секунды.
    """
    try:
        import osmnx as ox
        ox.settings.timeout = 30  # Таймаут Overpass API (секунды)
    except ImportError:
        print("   osmnx не установлен. Калибровка по перцентилю.")
        return _fallback_calibration(dem)
    
    all_water_elevations = []
    
    for lon, lat in seeds_wgs84:
        # Маленький bbox вокруг seed-точки (~2 км)
        dlat = buffer_km / 111.0                          # ~1 км в градусах широты
        dlon = buffer_km / (111.0 * np.cos(np.radians(lat)))  # с учётом широты
        
        small_bbox = (lon - dlon, lat - dlat, lon + dlon, lat + dlat)
        
        try:
            print(f"   Запрос воды вокруг ({lon:.2f}, {lat:.2f})...")
            water = ox.features_from_bbox(
                bbox=small_bbox,
                tags={'natural': 'water'}
            )
            if water.empty:
                continue
            
            water = water.to_crs(crs_metric)
            water = water[water.geometry.type.isin(['Polygon', 'MultiPolygon'])]
            if water.empty:
                continue
            
            # Растеризуем на сетку DEM
            water_mask = rasterize(
                [(geom, 1) for geom in water.geometry],
                out_shape=(dem.sizes['y'], dem.sizes['x']),
                transform=dem.rio.transform(),
                fill=0,
                dtype='uint8'
            )
            
            dem_values = dem.values[0]
            elevations = dem_values[water_mask == 1]
            elevations = elevations[~np.isnan(elevations)]
            elevations = elevations[(elevations > -9990) & (elevations != -9999)]  # убираем nodata
            elevations = elevations[(elevations > -2) & (elevations < 50)]   # физический диапазон WGS84 воды СПб
            
            if len(elevations) > 0:
                all_water_elevations.append(elevations)
                print(f"     Найдено {len(elevations)} водных пикселей")
                
        except Exception as e:
            print(f"     Ошибка: {e}")
            continue
    
    if all_water_elevations:
        combined = np.concatenate(all_water_elevations)
        if len(combined) >= 50:
            water_median = float(np.median(combined))
            print(f"   Медиана высот на воде (OSM): {water_median:.2f} м")
            print(f"   Стд. откл.: {np.std(combined):.2f} м")
            print(f"   Всего водных пикселей: {len(combined)}")
            return water_median
    
    print("   Недостаточно водных данных из OSM. Калибровка по перцентилю.")
    return _fallback_calibration(dem)


def _fallback_calibration(dem):
    """Резервная калибровка — улучшенная версия оригинала."""
    flat_vals = dem.values.flatten()
    valid = flat_vals[(~np.isnan(flat_vals)) & (flat_vals > -50) & (flat_vals < 50)]
    if len(valid) == 0:
        return 0
    water_base = np.percentile(valid, 5)
    print(f"   Резервная калибровка: 5-й перцентиль = {water_base:.2f} м")
    return water_base


# ======================================================================================
# 3. МАСКА ЗАТОПЛЕНИЯ С ГИДРОЛОГИЧЕСКОЙ СВЯЗНОСТЬЮ
# ======================================================================================
def compute_flood_mask(dem, flood_level, water_offset, seeds_wgs84, crs_metric,
                       morph_radius=3):
    """
    Строит маску затопления с проверкой гидрологической связности.
    
    1. Пороговая маска: все пиксели ≤ flood_level + offset
    2. Морфологическое закрытие — заполняет мелкие дыры
    3. Связные компоненты (connected components)
    4. Сохраняем только компоненты, содержащие seed-точки (Нева, залив)
    """
    effective_level = flood_level + water_offset
    print(f"   Эффективный порог: {effective_level:.2f} м")
    
    dem_arr = dem.values[0]
    
    # Шаг 1: пороговая маска
    raw_mask = (dem_arr <= effective_level) & (~np.isnan(dem_arr))
    raw_mask = raw_mask.astype(np.uint8)
    
    # Шаг 2: морфологическое закрытие
    if morph_radius > 0:
        struct = ndimage.generate_binary_structure(2, 2)
        raw_mask = ndimage.binary_closing(
            raw_mask, structure=struct, iterations=morph_radius
        ).astype(np.uint8)
    
    # Шаг 3: связные компоненты
    labels, num_features = ndimage.label(raw_mask)
    print(f"   Найдено связных компонент: {num_features}")
    
    # Шаг 4: определяем, какие компоненты содержат seed-точки
    transform = dem.rio.transform()
    valid_labels = set()
    
    for lon, lat in seeds_wgs84:
        seed_gdf = gpd.GeoDataFrame(
            {'geometry': [Point(lon, lat)]}, crs="EPSG:4326"
        ).to_crs(crs_metric)
        sx, sy = seed_gdf.geometry.iloc[0].x, seed_gdf.geometry.iloc[0].y
        
        # Пиксельные координаты
        col = int((sx - transform.c) / transform.a)
        row = int((sy - transform.f) / transform.e)
        
        if 0 <= row < labels.shape[0] and 0 <= col < labels.shape[1]:
            label_val = labels[row, col]
            if label_val > 0:
                valid_labels.add(label_val)
                print(f"   Seed ({lon:.2f}, {lat:.2f}) → компонента {label_val}")
            else:
                print(f"   ⚠ Seed ({lon:.2f}, {lat:.2f}) не попал в маску затопления")
        else:
            print(f"   ⚠ Seed ({lon:.2f}, {lat:.2f}) вне границ DEM")
    
    if not valid_labels:
        print("   ⚠ Ни один seed не попал в маску. Берём крупные компоненты (>10000 px).")
        for lbl in range(1, num_features + 1):
            if np.sum(labels == lbl) > 10000:
                valid_labels.add(lbl)
    
    if not valid_labels:
        print("   ⚠ Нет крупных компонент. Возвращаем всю маску.")
        return raw_mask
    
    # Финальная маска
    flood_mask = np.isin(labels, list(valid_labels)).astype(np.uint8)
    
    n_total = int(np.sum(raw_mask))
    n_connected = int(np.sum(flood_mask))
    n_removed = n_total - n_connected
    print(f"   Пикселей до фильтрации: {n_total:,}")
    print(f"   Пикселей после фильтрации: {n_connected:,}")
    if n_total > 0:
        print(f"   Удалено изолированных: {n_removed:,} ({100*n_removed/n_total:.1f}%)")
    
    return flood_mask


# ======================================================================================
# 4. ВЕКТОРИЗАЦИЯ С ФИЛЬТРАЦИЕЙ
# ======================================================================================
def vectorize_flood(flood_mask, transform, crs, min_area_m2, simplify_m):
    """Векторизует маску, фильтрует мелкие полигоны, упрощает В МЕТРАХ."""
    from shapely.geometry import shape as _sh
    geoms = [s for s, v in shapes(flood_mask, mask=flood_mask == 1, transform=transform)]
    if not geoms:
        return None
    # rasterio.shapes() returns GeoJSON dicts in raster CRS — convert to shapely explicitly
    flood_gdf = gpd.GeoDataFrame(geometry=[_sh(g) for g in geoms], crs=crs)
    print(f"   Полигонов до фильтрации: {len(flood_gdf)}")
    
    # Фильтр по площади
    flood_gdf = flood_gdf[flood_gdf.geometry.area >= min_area_m2]
    print(f"   Полигонов после фильтрации (≥{min_area_m2} м²): {len(flood_gdf)}")
    
    if flood_gdf.empty:
        return None
    
    flood_gdf = flood_gdf.dissolve()
    
    # ИСПРАВЛЕНИЕ: упрощение В МЕТРИЧЕСКОЙ CRS, не после to_crs(4326)!
    flood_gdf.geometry = flood_gdf.geometry.simplify(simplify_m)
    
    total_area_ha = flood_gdf.geometry.area.sum() / 10000
    print(f"   Площадь затопления: {total_area_ha:.1f} га")
    
    return flood_gdf


# ======================================================================================
# 5. АНАЛИЗ УЩЕРБА
# ======================================================================================
def analyze_impact(flood_gdf, bbox_wgs84, crs_metric):
    """
    Анализ landuse + зданий в зоне затопления.
    
    ИСПРАВЛЕНИЕ v2.1: используем bbox зоны затопления (а не всего DEM),
    что кратно уменьшает объём запроса к Overpass API.
    """
    try:
        import osmnx as ox
        ox.settings.timeout = 60
    except ImportError:
        print("   osmnx не установлен. Пропуск анализа ущерба.")
        return {}
    
    # Bbox зоны затопления (а не всего DEM — намного меньше!)
    flood_wgs = flood_gdf.to_crs("EPSG:4326")
    fb = flood_wgs.total_bounds  # minx, miny, maxx, maxy
    bbox_tuple = (fb[0], fb[1], fb[2], fb[3])
    
    # Проверка размера bbox (защита от слишком больших запросов)
    area_deg2 = (fb[2] - fb[0]) * (fb[3] - fb[1])
    print(f"   Bbox зоны затопления: {area_deg2:.3f} deg² ({fb})")
    
    results = {}
    
    # --- Landuse ---
    try:
        print("   Загрузка landuse из OSM...")
        landuse = ox.features_from_bbox(bbox=bbox_tuple, tags={'landuse': True})
        if not landuse.empty:
            landuse = landuse.to_crs(crs_metric)
            landuse = landuse[landuse.geometry.type.isin(['Polygon', 'MultiPolygon'])]
            impact_lu = gpd.overlay(landuse, flood_gdf, how='intersection')
            impact_lu['flooded_ha'] = impact_lu.geometry.area / 10000
            results['landuse'] = impact_lu
            
            print("\n   ТОП ПОСТРАДАВШИХ ЗОН (landuse):")
            summary = impact_lu.groupby('landuse')['flooded_ha'].sum().sort_values(ascending=False)
            print(summary.head(10).to_string())
    except Exception as e:
        print(f"   Ошибка загрузки landuse: {e}")
    
    # --- Здания (только если зона не слишком большая) ---
    if area_deg2 > 0.5:
        print(f"   ⚠ Зона затопления слишком велика ({area_deg2:.1f} deg²) для загрузки зданий.")
        print("     Пропускаем слой зданий. Уменьшите bbox или повысьте FLOOD_LEVEL.")
    else:
        try:
            print("\n   Загрузка зданий из OSM...")
            buildings = ox.features_from_bbox(bbox=bbox_tuple, tags={'building': True})
            if not buildings.empty:
                buildings = buildings.to_crs(crs_metric)
                buildings = buildings[buildings.geometry.type.isin(['Polygon', 'MultiPolygon'])]
                flooded_buildings = gpd.sjoin(buildings, flood_gdf, how='inner', predicate='intersects')
                results['buildings'] = flooded_buildings
                print(f"   Зданий в зоне затопления: {len(flooded_buildings)}")
        except Exception as e:
            print(f"   Ошибка загрузки зданий: {e}")
    
    return results


# ======================================================================================
# 6. ВИЗУАЛИЗАЦИЯ
# ======================================================================================
def build_map(flood_gdf, impact_results, bbox_wgs84, output_path):
    """Строит интерактивную карту Folium."""
    b_minx, b_miny, b_maxx, b_maxy = bbox_wgs84
    cy, cx = (b_miny + b_maxy) / 2, (b_minx + b_maxx) / 2
    
    m = folium.Map(location=[cy, cx], zoom_start=12, tiles='CartoDB positron')
    
    # Перепроекция в WGS84 для Folium (упрощение уже сделано в метрах!)
    flood_wgs = flood_gdf.to_crs("EPSG:4326")
    
    folium.GeoJson(
        flood_wgs,
        name='Зона затопления',
        style_function=lambda x: {
            'fillColor': '#1E90FF', 'color': '#0000CD',
            'weight': 0.5, 'fillOpacity': 0.5
        }
    ).add_to(m)
    
    if 'landuse' in impact_results and not impact_results['landuse'].empty:
        impact_wgs = impact_results['landuse'].to_crs("EPSG:4326")
        folium.GeoJson(
            impact_wgs,
            name='Затопленные территории',
            style_function=lambda x: {
                'fillColor': '#FF4500', 'color': 'black',
                'weight': 0.5, 'fillOpacity': 0.5
            },
            tooltip=folium.GeoJsonTooltip(
                fields=['landuse', 'flooded_ha'],
                aliases=['Тип', 'Площадь (га)'],
                localize=True
            )
        ).add_to(m)
    
    if 'buildings' in impact_results and not impact_results['buildings'].empty:
        bld_wgs = impact_results['buildings'].to_crs("EPSG:4326")
        if len(bld_wgs) > 5000:
            print(f"   ⚠ Много зданий ({len(bld_wgs)}), показываем 5000")
            bld_wgs = bld_wgs.head(5000)
        
        folium.GeoJson(
            bld_wgs[['geometry']],
            name='Затопленные здания',
            style_function=lambda x: {
                'fillColor': '#FF0000', 'color': '#8B0000',
                'weight': 0.3, 'fillOpacity': 0.7
            },
            show=False  # Скрыт по умолчанию
        ).add_to(m)
    
    folium.LayerControl().add_to(m)
    m.save(output_path)
    print(f"   Карта сохранена: {output_path}")
    return m


# ======================================================================================
# MAIN
# ======================================================================================
def main():
    print(f"{'='*70}")
    print(f" АНАЛИЗ ЗАТОПЛЕНИЯ: {CITY_NAME}")
    print(f" Уровень подъёма воды: {FLOOD_LEVEL} м")
    print(f"{'='*70}")
    
    # 0. Загрузка тайлов
    print("\n[0/5] Загрузка тайлов ArcticDEM...")
    tif_files = download_dem_tiles(TILE_SHORT_IDS, COLLECTION_ID, STAC_API_URL, OUTPUT_DIR)
    
    # 1. Объединение и репроекция
    print("\n[1/5] Объединение и репроекция DEM...")
    merge_and_reproject(tif_files, CRS_METRIC, TARGET_RES, VRT_PATH, INTERMEDIATE_FILE)
    
    # 2. Загрузка и калибровка
    print("\n[2/5] Загрузка DEM и калибровка...")
    dem = rioxarray.open_rasterio(INTERMEDIATE_FILE, masked=True)
    
    xmin, ymin, xmax, ymax = dem.rio.bounds()
    bbox_poly = gpd.GeoDataFrame(
        {'geometry': [box(xmin, ymin, xmax, ymax)]}, crs=CRS_METRIC
    )
    bbox_wgs84 = bbox_poly.to_crs("EPSG:4326").total_bounds
    print(f"   Bbox WGS84: {bbox_wgs84}")
    
    water_offset = calibrate_elevation(dem, bbox_wgs84, CRS_METRIC, WATER_SEEDS_WGS84)
    
    # 3. Маска затопления
    print(f"\n[3/5] Расчёт маски затопления ({FLOOD_LEVEL} м + калибровка)...")
    flood_mask = compute_flood_mask(
        dem, FLOOD_LEVEL, water_offset, WATER_SEEDS_WGS84, CRS_METRIC,
        morph_radius=MORPH_CLOSING_RADIUS
    )
    
    # 4. Векторизация
    print("\n[4/5] Векторизация...")
    flood_gdf = vectorize_flood(
        flood_mask, dem.rio.transform(), CRS_METRIC,
        MIN_POLYGON_AREA_M2, SIMPLIFY_TOLERANCE_M
    )
    
    if flood_gdf is None or flood_gdf.empty:
        print("Затопления не обнаружено при заданном уровне.")
        return
    
    # 5. Анализ и карта
    print("\n[5/5] Анализ ущерба и построение карты...")
    impact = analyze_impact(flood_gdf, bbox_wgs84, CRS_METRIC)
    m = build_map(flood_gdf, impact, bbox_wgs84, OUT_FILENAME)
    
    print(f"\n{'='*70}")
    print(f" ГОТОВО!")
    print(f" Карта: {OUT_FILENAME}")
    print(f"{'='*70}")
    
    return m


# Запуск
main()

 АНАЛИЗ ЗАТОПЛЕНИЯ: Saint Petersburg, Russia
 Уровень подъёма воды: 3.05 м

[0/5] Загрузка тайлов ArcticDEM...
   Коллекция: arcticdem-mosaics-v4.1-2m
   Целевые тайлы: ['32_73_1_1_2m_v4.1', '32_73_2_1_2m_v4.1']
   Найдено тайлов: 2
   Загрузка 32_73_2_1_2m_v4.1...
   Сохранено: /kaggle/working/32_73_2_1_2m_v4.1.tif
   Загрузка 32_73_1_1_2m_v4.1...
   Сохранено: /kaggle/working/32_73_1_1_2m_v4.1.tif

[1/5] Объединение и репроекция DEM...
   Тайлов для объединения: 2
     - 32_73_2_1_2m_v4.1.tif
     - 32_73_1_1_2m_v4.1.tif
   Создание VRT...
   Репроекция в EPSG:32636, разрешение 2 м...
   DEM готов.

[2/5] Загрузка DEM и калибровка...
   Bbox WGS84: [29.26832631 59.56986184 31.18348311 60.21693898]
   Запрос воды вокруг (30.25, 59.95)...
     Найдено 1532986 водных пикселей
   Запрос воды вокруг (30.35, 59.94)...
     Найдено 8361458 водных пикселей
   Запрос воды вокруг (30.40, 59.95)...
     Найдено 8085642 водных пикселей
   Медиана высот на воде (OSM): 17.83 м
   Стд. откл.: 1.80 